# Week 2 · Tutorial 2 — Outliers, mean and variance, median and IQR

**Covers videos 2.3 – 2.5:** outliers (*why* is a point far out?) · mean and variance · the robust pair median/IQR.

**Datasets**
* `river_inflow.csv` — daily inflow (m³/s) into a hydropower reservoir, 2020–2022. Right-skewed; contains a few absurd readings.
* `wind_turbine_scada.csv` — hourly wind speed (m/s) and power (kW) of one wind turbine, 2022. Rated power 2 000 kW.

A point can be far out for three reasons: it is **wrong** (sensor, unit), it is **genuine but rare** (a flood), or it is
**ordinary only in combination with other variables** (high power at low wind speed). The treatment is different each time.

### How to work with this notebook (no programming experience needed)

* A notebook is a list of **cells**. Click a code cell and press **Shift + Enter** to run it; the result appears below it. Run the cells **from top to bottom**.
* Cells that start with `import` or read a file (`pd.read_csv(...)`) just load tools and data. You do not need to understand every line.
* In every **✏️ Your turn** block you make one small, exactly described change (a number, a word, or a line to paste in), then run the cell again.
  The box **What to change** shows the exact text to type. Lines marked `# TODO` are the ones to edit.
* Write down the numbers that appear — the quiz asks for them. If you break something, re-run the cells above it, or use *Kernel → Restart & Run All*.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/blasern/data-tutorials/main/data/"   # where the data files live
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

## 1. Absurd values: use physical limits

The maximum of the file is far above anything a river of this size can produce.

In [ ]:
river = pd.read_csv(DATA_URL + "river_inflow.csv", parse_dates=["date"])
print(river.shape)
print(river["inflow_m3s"].describe().round(1))
print()
print("five largest:", river["inflow_m3s"].nlargest(5).tolist())
print("smallest    :", river["inflow_m3s"].nsmallest(2).tolist())

### ✏️ Your turn 1

Define a **physically plausible range** and set values outside it to missing (`NaN`) — the usual default, we do not delete rows.

The starter range accepts everything. Change it to `valid_min = 0` and `valid_max = 5000` (m³/s). The cell then reports how
many readings were removed, and the new maximum — a genuine flood.

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
valid_min = -1e9
valid_max = 1e9
```
by
```python
valid_min = 0
valid_max = 5000
```

In [ ]:
valid_min = -1e9      # TODO: 0
valid_max = 1e9       # TODO: 5000

bad = (river["inflow_m3s"] < valid_min) | (river["inflow_m3s"] > valid_max)
river["inflow_clean"] = river["inflow_m3s"].mask(bad)

print("readings outside the range:", bad.sum())
print("maximum after cleaning:", river["inflow_clean"].max())
print("mean raw:", round(river["inflow_m3s"].mean(), 1), "| mean clean:", round(river["inflow_clean"].mean(), 1))
print("std  raw:", round(river["inflow_m3s"].std(), 1), "| std  clean:", round(river["inflow_clean"].std(), 1))

## 2. Two standard rules disagree

After removing the absurd values, which of the remaining points are "outliers"? Two standard rules:

* **z-score:** flag points more than `z` standard deviations from the mean.
* **IQR rule:** flag points more than `k` × IQR below Q1 or above Q3.

For a right-skewed variable like river inflow, they give very different answers.

### ✏️ Your turn 2

Count the flagged points (on `river["inflow_clean"]`, ignoring `NaN`) with the rules as given (`z = 3`, `k = 1.5`).
Then change **both** values at the `TODO` marks to `z = 4` and `k = 3` and run again. The cell also reports how many points
are flagged by *both* rules.

In [ ]:
x = river["inflow_clean"].dropna()

z = 3        # TODO: run again with z = 4
k = 1.5      # TODO: run again with k = 3

zscore = (x - x.mean()) / x.std()
q1, q3 = x.quantile(0.25), x.quantile(0.75)
iqr = q3 - q1

flag_z = zscore.abs() > z
flag_iqr = (x < q1 - k * iqr) | (x > q3 + k * iqr)

print(f"z-score rule (z={z}):  {flag_z.sum()} flagged")
print(f"IQR rule     (k={k}): {flag_iqr.sum()} flagged")
print("IQR =", round(iqr, 1), "| Q1 =", round(q1, 1), "| Q3 =", round(q3, 1))
print("flagged by both:", (flag_z & flag_iqr).sum())

## 3. Look at two columns at once

In `wind_turbine_scada.csv` no single column has an impossible value: power is always between 0 and 2 050 kW, wind speed
between 1 and 18 m/s. But **power and wind speed together** must follow the power curve. A reading of 1 500 kW at 4 m/s is
wrong, even though both numbers are individually fine. Meanwhile, **0 kW at 14 m/s** can be genuine: the grid operator
curtailed the turbine (column `curtailed` = 1).

In [ ]:
scada = pd.read_csv(DATA_URL + "wind_turbine_scada.csv", parse_dates=["timestamp"])
print(scada.shape, "| curtailed hours:", scada["curtailed"].sum())

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(scada["wind_speed_ms"], scada["power_kw"], s=3, alpha=0.4)
ax.set_xlabel("wind speed (m/s)"); ax.set_ylabel("power (kW)")
plt.show()

We estimate the power curve as the **median power in each 1 m/s wind-speed bin**, and look at the *residual*: the power
minus the expected power at that wind speed.

In [ ]:
bin_width = 1.0
scada["bin"] = (scada["wind_speed_ms"] // bin_width) * bin_width
expected = scada.groupby("bin")["power_kw"].transform("median")
scada["residual"] = scada["power_kw"] - expected
scada["residual"].describe().round(1)

### ✏️ Your turn 3

Flag residuals larger than a threshold.

1. Change `threshold` from 3000 to **800** (kW).
2. Fill in the `TODO` lines: `too_high` = flagged rows with a **positive** residual (more power than the curve allows);
   `too_low` = flagged rows with a **negative** residual; and `too_low_genuine` = the `too_low` rows that have `curtailed == 1`.
3. Rows in `too_high` are wrong → set their `power_kw` to NaN in the new column `power_clean`; the others stay.

**What to change** — edit the cell exactly like this, then run it:

1. Change `threshold = 3000` into `threshold = 800`

2. Change `too_high = flagged & (scada["residual"] > 0)` into `too_high = flagged & (scada["residual"] > 0)`

In [ ]:
threshold = 3000      # TODO: 800

flagged = scada["residual"].abs() > threshold
too_high = flagged & (scada["residual"] > 0)       # TODO: check that this is what is described
too_low = flagged & (scada["residual"] < 0)
too_low_genuine = too_low & (scada["curtailed"] == 1)

scada["power_clean"] = scada["power_kw"].mask(too_high)

print("flagged:", flagged.sum())
print("too high (positive residual):", too_high.sum())
print("too low (negative residual):", too_low.sum(), "| of which curtailed:", too_low_genuine.sum())
print("mean power raw  :", round(scada["power_kw"].mean(), 2))
print("mean power clean:", round(scada["power_clean"].mean(), 2))

The `too_low` hours are not errors — the grid operator really did stop the turbine. They stay in the data (perhaps with a
flag). *Whether to keep, correct, set to missing or delete depends on the reason, not on the value.*

## 4. Mean, variance — and what a skewed variable hides

The mean is pulled towards the tail. For river inflow the mean is far above the typical day.

### ✏️ Your turn 4

Using the cleaned inflow series `x = river["inflow_clean"].dropna()`, fill in the `TODO`s (2 decimals unless stated):
- `mean_x`, `median_x`
- `skew_x`: the skewness (`x.skew()`)
- `share_above_mean`: the share of days with inflow **above the mean** (3 decimals)
- `cv`: coefficient of variation = standard deviation / mean

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
mean_x = np.nan
median_x = np.nan
skew_x = np.nan
share_above_mean = np.nan
cv = np.nan
```
by
```python
mean_x = x.mean()
median_x = x.median()
skew_x = x.skew()
share_above_mean = (x > mean_x).mean()
cv = x.std() / x.mean()
```

In [ ]:
x = river["inflow_clean"].dropna()

mean_x = np.nan             # TODO
median_x = np.nan           # TODO
skew_x = np.nan             # TODO
share_above_mean = np.nan   # TODO
cv = np.nan                 # TODO

print("mean / median:", round(mean_x, 2), round(median_x, 2))
print("skew:", round(skew_x, 2))
print("share of days above the mean:", round(share_above_mean, 3))
print("coefficient of variation:", round(cv, 2))

## 5. Median and IQR — robust to *how extreme*, not to *how many*

We contaminate the clean series with `k` absurd readings of 99 999 m³/s at random days, and see which statistic moves.

### ✏️ Your turn 5

Change `k` from 1 to **5** and run the cell. Compare the statistics before and after the contamination.

**What to change** — edit the cell exactly like this, then run it:

Change `k = 1` into `k = 5`

In [ ]:
k = 1      # TODO: 5

rng = np.random.default_rng(1)
x = river["inflow_clean"].dropna().reset_index(drop=True)
x_bad = x.copy()
x_bad.iloc[rng.choice(len(x), size=k, replace=False)] = 99999.0

def stats(s):
    return pd.Series({"mean": s.mean(), "std": s.std(), "median": s.median(), "IQR": s.quantile(0.75) - s.quantile(0.25)})

comparison = pd.DataFrame({"before": stats(x), "after": stats(x_bad)}).round(1)
print(comparison)

### The tail is where the engineering is

The median ignores the tail, but for a reservoir the tail *is* the point: spillway design is based on rare, large inflows.

### ✏️ Your turn 6

Using the cleaned series `x`, compute (1 decimal):
- `q95`, `q99`: the 95 % and 99 % quantiles of the daily inflow (`x.quantile(0.95)`),
- `days_over_q99`: the number of days with inflow above `q99`,
- `tail_volume_share`: the share of the **total** inflow volume (sum of all days) delivered by the days *above q95* (3 decimals).

**What to change** — edit the cell exactly like this, then run it:

Replace
```python
q95 = np.nan
q99 = np.nan
days_over_q99 = np.nan
tail_volume_share = np.nan
```
by
```python
q95 = x.quantile(0.95)
q99 = x.quantile(0.99)
days_over_q99 = int((x > q99).sum())
tail_volume_share = x[x > q95].sum() / x.sum()
```

In [ ]:
q95 = np.nan                  # TODO
q99 = np.nan                  # TODO
days_over_q99 = np.nan        # TODO
tail_volume_share = np.nan    # TODO

print("q95, q99:", round(q95, 1), round(q99, 1))
print("days above q99:", days_over_q99)
print("share of volume from days above q95:", round(tail_volume_share, 3))
print("median for comparison:", round(x.median(), 1))

## Summary

* An outlier is not about *whether* a value is extreme, but *why*. Check physical limits, then relationships between columns.
* The standard rules (z-score, IQR) disagree for skewed data; neither knows what is physically possible.
* Mean and standard deviation are dragged by a handful of absurd readings; median and IQR are not — but a robust statistic is not the same as a correct one when the tail is the point.